# XGBoost 回归 — 完整流程 Demo

本 notebook 演示使用 `ml_tool` 工具包完成 XGBoost 回归模型的端到端流程：

1. 初始化数据 & 数据集划分
2. 特征分析（缺失率 / 一值率 / 分位数 / PSI / 相关性）
3. 特征筛选（缺失率 + PSI + 相关性，回归跳过 IV）
4. XGBoost 回归模型训练（Hyperopt 自动调参）
5. 自定义超参数搜索空间（可选）
6. 模型评估报告（指标 + 分桶分析 + 特征重要性）
7. 特征分析报告
8. 模型保存与加载

In [1]:
import pandas as pd
import numpy as np
from ml_tool import FeatureAnalyzer, FeatureSelector, ModelTrainer, ReportGenerator, split_dataset

np.random.seed(42)
n = 3000
dates = pd.date_range('2024-01-01', periods=n, freq='D').to_series().sample(frac=1, random_state=42).values
df_raw = pd.DataFrame({
    'f1': np.random.randn(n),
    'f2': np.random.rand(n),
    'f3': np.random.randn(n) * 2,
    'f4': np.random.rand(n),
    'f5': np.random.randn(n),
    'y_reg': 2.5 * np.random.randn(n) - 1.2 * np.random.rand(n) + np.random.randn(n) * 1.5,
    'report_date': dates,
})
df_raw.loc[df_raw.sample(frac=0.05).index, 'f2'] = np.nan

df = split_dataset(df_raw, date_col='report_date', oot_date='2024-09-01', train_ratio=0.8)
df['month'] = df['report_date'].dt.to_period('M').astype(str)
print(df.shape, df['dataset'].value_counts().sort_index().to_dict())
print('目标变量统计:')
print(df['y_reg'].describe())

OUTPUT_DIR = './output/xgb_reg'
# 模拟附加列（在 split_dataset 之后追加）
df['score_A']    = np.random.uniform(300, 700, len(df))   # 模拟标品A分
df['score_B']    = np.random.uniform(350, 750, len(df))   # 模拟标品B分
df['gain_score'] = np.random.uniform(0, 1, len(df))       # 模拟外部增益分
df['overdue_30'] = np.random.binomial(1, 0.12, len(df))   # 模拟逾期30天标签
df['overdue_60'] = np.random.binomial(1, 0.08, len(df))   # 模拟逾期60天标签

数据集划分完成（切点: 2024-09-01, train_ratio=0.8）
  train:    195  (6.5%)
  test :     49  (1.6%)
  oot  :   2756  (91.9%)
(3000, 9) {'oot': 2756, 'test': 49, 'train': 195}
目标变量统计:
count    3000.000000
mean       -0.566390
std         2.970119
min       -11.544836
25%        -2.605768
50%        -0.631346
75%         1.416308
max        11.459538
Name: y_reg, dtype: float64


## 1. 特征分析

In [2]:
feature_cols = ['f1', 'f2', 'f3', 'f4', 'f5']
analyzer = FeatureAnalyzer(df[feature_cols])

print('=== 缺失率 ==='); display(analyzer.missing_rate())
print('=== 一值率 ==='); display(analyzer.single_value_rate())
print('=== 分位数统计 ==='); display(analyzer.quantile_stats())

base_df    = df[df['dataset'] == 'train'][feature_cols]
compare_df = df[df['dataset'] == 'oot'][feature_cols]
psi_df = analyzer.psi(base_df, compare_df)
print('=== PSI（train vs OOT）==='); display(psi_df)
print('=== 特征间相关性（|r|>0.3）==='); display(analyzer.correlation(threshold=0.3))


=== 缺失率 ===


,特征名,缺失数,缺失率
0,f1,0,0.00
1,f2,150,0.05
2,f3,0,0.00
3,f4,0,0.00
4,f5,0,0.00


=== 一值率 ===


,特征名,最高频值,一值率
0,f1,0.496714,0.0003
1,f2,NaN,0.0500
2,f3,-0.380482,0.0003
3,f4,0.648747,0.0003
4,f5,1.508311,0.0003


=== 分位数统计 ===


,特征名,样本数,均值,标准差,最小值,1%分位,5%分位,25%分位,中位数,75%分位,95%分位,99%分位,最大值
0,f1,3000.0,0.032001,0.986808,-3.241267,-2.219490,-1.565074,-0.627541,0.024365,0.673591,1.673504,2.364268,3.926238
1,f2,2850.0,0.494762,0.287839,0.000031,0.013254,0.054535,0.246425,0.486030,0.742287,0.950403,0.988396,0.999461
2,f3,3000.0,-0.035827,2.080612,-7.673311,-4.769815,-3.475749,-1.440090,-0.065302,1.397700,3.363288,4.994851,7.058110
3,f4,3000.0,0.499154,0.291073,0.000158,0.008810,0.050133,0.244087,0.490364,0.756368,0.952307,0.990111,0.999673
4,f5,3000.0,0.012466,0.997329,-3.375579,-2.342613,-1.606774,-0.651083,0.015079,0.673944,1.612335,2.384736,3.428910


=== PSI（train vs OOT）===


,特征名,PSI
0,f1,0.0210
1,f2,0.0422
2,f3,0.0372
3,f4,0.0667
4,f5,0.0581


=== 特征间相关性（|r|>0.3）===


,特征A,特征B,相关系数


## 2. 特征筛选（回归）

In [3]:
import os

reporter = ReportGenerator(output_dir=OUTPUT_DIR)
train_df = df[df['dataset'] == 'train'].reset_index(drop=True)

# 回归模式：psi_filter=False corr_filter=False，PSI/相关性只展示不剔除
# iv_threshold=0 跳过 IV 筛选
selector = FeatureSelector(
    iv_threshold=0.0,
    psi_threshold=0.5,
    missing_threshold=0.98,
    corr_threshold=0.97,
    psi_filter=False,
    corr_filter=False,
)
selector.fit(
    df=train_df[feature_cols + ['y_reg']],
    target='y_reg',
    base_df=base_df,
    compare_df=compare_df,
)
print('入选特征:', selector.get_selected())
print('剔除特征:', selector.dropped_cols)
display(selector.get_report())

analysis = {
    '缺失率': analyzer.missing_rate(),
    '分位数统计': analyzer.quantile_stats(),
    'PSI': psi_df,
    '相关性': analyzer.correlation(threshold=0.0),
}
path = reporter.feature_selection_report(selector.get_report(), analysis_results=analysis)
print('特征筛选报告:', path)


入选特征: ['f1', 'f2', 'f3', 'f4', 'f5']
剔除特征: []


C:\miniconda\lib\site-packages\pandas\core\arraylike.py:396: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\miniconda\lib\site-packages\pandas\core\arraylike.py:396: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\miniconda\lib\site-packages\pandas\core\arraylike.py:396: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\miniconda\lib\site-packages\pandas\core\arraylike.py:396: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\miniconda\lib\site-packages\pandas\core\arraylike.py:396: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,特征名,缺失率,PSI,IV,是否保留,剔除原因
0,f1,0.0000,0.0210,inf,保留,
1,f2,0.0256,0.0422,inf,保留,
2,f3,0.0000,0.0372,inf,保留,
3,f4,0.0000,0.0667,inf,保留,
4,f5,0.0000,0.0581,inf,保留,


特征筛选报告: ./output/xgb_reg\02_feature_selection\特征筛选报告.xlsx


## 3. 模型训练（XGBoost 回归）

In [4]:
selected = selector.get_selected()

X_train = df[df['dataset'] == 'train'][selected]
y_train = df[df['dataset'] == 'train']['y_reg']
X_test  = df[df['dataset'] == 'test'][selected]
y_test  = df[df['dataset'] == 'test']['y_reg']
X_oot   = df[df['dataset'] == 'oot'][selected]
y_oot   = df[df['dataset'] == 'oot']['y_reg']

# n_trials 实际建议 100，这里用 20 演示
trainer = ModelTrainer(model_type='xgboost_reg', n_trials=20)
trainer.fit(
    X_train, y_train,
    X_test,  y_test,
    X_oot,   y_oot,
    save_dir=OUTPUT_DIR,
)
print('入模特征数:', len(trainer.selected_features))
print('入模特征: ', trainer.selected_features)
print('最优参数: ', trainer.best_params)


XGB回归 第1轮（固定参数训练）


XGB回归 第2轮:   0%|                                               | 0/20 [00:00<?, ?it/s]

调参日志已保存至: ./output/xgb_reg\03_model_tuning\tuning_log.xlsx
模型已保存至: ./output/xgb_reg\05_model_deploy
入模特征数: 5
入模特征:  ['f4', 'f2', 'f1', 'f3', 'f5']
最优参数:  {'colsample_bytree': 0.8, 'gamma': 7.0, 'learning_rate': 0.23, 'max_depth': 7, 'min_child_weight': 9.0, 'reg_alpha': 18.0, 'reg_lambda': 17.0, 'subsample': 0.9, 'objective': 'reg:squarederror', 'booster': 'gbtree', 'eval_metric': 'mae', 'n_estimators': 800, 'random_state': 2024, 'tree_method': 'hist', 'early_stopping_rounds': 30, 'nthread': -1}


In [5]:
print('调参日志（按 loss 升序，前 10 行）:')
display(
    trainer.trials_log[['round', 'trial', 'loss', 'is_best', 'train_mae', 'test_mae', 'oot_mae']]
    .sort_values('loss').head(10)
)


调参日志（按 loss 升序，前 10 行）:


,round,trial,loss,is_best,train_mae,test_mae,oot_mae
10,第2轮,10,2.946402,True,2.3679,2.0779,2.3949
1,第2轮,1,2.947863,False,2.3516,2.0685,2.3923
2,第2轮,2,2.949261,False,2.3689,2.0789,2.3973
19,第2轮,19,2.949261,False,2.3689,2.0789,2.3973
8,第2轮,8,2.949261,False,2.3689,2.0789,2.3973
7,第2轮,7,2.950872,False,2.3581,2.0763,2.3965
16,第2轮,16,2.952889,False,2.3451,2.0884,2.3974
0,第1轮,1,2.953597,False,2.3454,2.0744,2.3962
3,第2轮,3,2.955423,False,2.3380,2.0848,2.3976
12,第2轮,12,2.956463,False,2.3345,2.0697,2.3960


## 4. 自定义超参数空间（可选）

In [6]:
from hyperopt import hp

# trainer.get_default_params / get_default_space ä¸éè¦ n_data
params1 = trainer.get_default_params()
print('第1阶固定参数:')
for k, v in params1.items():
    print(f'  {k:30s}: {v}')

space = trainer.get_default_space()
print('第2阶超参数空间 key:')
for k, v in space.items():
    print(f'  {k:30s}: {v}')


第1阶固定参数:
  objective                     : reg:squarederror
  booster                       : gbtree
  eval_metric                   : mae
  n_estimators                  : 800
  learning_rate                 : 0.05
  max_depth                     : 6
  subsample                     : 0.8
  colsample_bytree              : 0.8
  min_child_weight              : 5
  gamma                         : 2
  reg_alpha                     : 5
  reg_lambda                    : 5
  early_stopping_rounds         : 30
  nthread                       : -1
第2阶超参数空间 key:
  objective                     : reg:squarederror
  booster                       : gbtree
  eval_metric                   : mae
  learning_rate                 : 0 float
1   hyperopt_param
2     Literal{learning_rate}
3     quniform
4       Literal{0.01}
5       Literal{0.3}
6       Literal{0.01}
  gamma                         : 0 float
1   hyperopt_param
2     Literal{gamma}
3     quniform
4       Literal{0}
5       Literal{10}
6   

In [7]:
# 自定义第1轮固定参数
custom_p1 = {
    'max_depth': 5,
    'learning_rate': 0.03,
    'min_child_weight': 3,
    'gamma': 1,
    'n_estimators': 600,
}

# 自定义第2轮搜索空间（覆盖部分 key）
space['learning_rate']    = hp.quniform('learning_rate', 0.005, 0.05, 0.005)
space['max_depth']        = hp.choice('max_depth', [4, 5, 6, 7])
space['min_child_weight'] = hp.quniform('min_child_weight', 1, 20, 1)
space['subsample']        = hp.quniform('subsample', 0.7, 1.0, 0.1)

trainer_custom = ModelTrainer(model_type='xgboost_reg', n_trials=10)
trainer_custom.fit(
    X_train, y_train, X_test, y_test, X_oot, y_oot,
    custom_params=custom_p1,
    custom_space=space,
)
print('最终 learning_rate:', trainer_custom.best_params.get('learning_rate'))
print('最终 max_depth:    ', trainer_custom.best_params.get('max_depth'))
print('最终 subsample:    ', trainer_custom.best_params.get('subsample'))
print('最终 n_estimators: ', trainer_custom.best_params.get('n_estimators'))


XGB回归 第1轮（固定参数训练）


XGB回归 第2轮:   0%|                                               | 0/10 [00:00<?, ?it/s]

最终 learning_rate: 0.035
最终 max_depth:     5
最终 subsample:     0.9
最终 n_estimators:  800


## 5. 模型评估报告

In [8]:
import pandas as pd
# XGBoost 特征重要性从 booster.get_score 取
imp_dict = trainer._trainer.model.get_booster().get_score(importance_type='gain')
fi_df = pd.DataFrame(list(imp_dict.items()), columns=['feature', 'importance'])

month_data = {
    '训练集': df[df['dataset'] == 'train'][['month']].reset_index(drop=True),
    '验证集': df[df['dataset'] == 'test'][['month']].reset_index(drop=True),
    'OOT':   df[df['dataset'] == 'oot'][['month']].reset_index(drop=True),
}
datasets_reg = {
    '训练集': (y_train.values, trainer.predict(X_train)),
    '验证集': (y_test.values,  trainer.predict(X_test)),
    'OOT':   (y_oot.values,   trainer.predict(X_oot)),
}

import numpy as np
label_col_data = {
    '训练集': (y_train.values > 0).astype(int),
    '验证集': (y_test.values  > 0).astype(int),
    'OOT':   (y_oot.values   > 0).astype(int),
}

reg_result = reporter.regression_report(
    datasets_reg,
    filename='XGBoost_回归模型评估报告',
    n_bins=10,
    month_col_data=month_data,
    label_col_data=label_col_data,
    feature_importance=fi_df,
    raw_df=df,
    target_col='y_reg',
    gain_score_col='gain_score', gain_label_col='overdue_30',
    scorecard_cols=['score_A', 'score_B'],
    scorecard_label_cols=['overdue_30', 'overdue_60'],
)
display(reg_result['summary'])
print('Excel:', reg_result['excel_path'])
print('HTML: ', reg_result['html_path'])

C:\Data\工作\AI_model\ml_tool\report.py:1374: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_r,  pearson_p  = pearsonr(y_true, y_pred)
C:\Data\工作\AI_model\ml_tool\report.py:1375: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_r, spearman_p = spearmanr(y_true, y_pred)
C:\Data\工作\AI_model\ml_tool\report.py:1374: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_r,  pearson_p  = pearsonr(y_true, y_pred)
C:\Data\工作\AI_model\ml_tool\report.py:1375: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_r, spearman_p = spearmanr(y_true, y_pred)
C:\Data\工作\AI_model\ml_tool\report.py:1374: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_r,  pearson_p  = pearsonr(y_true, y_pred)
C:\Data\工作\AI_model\ml_tool\report.py:1375: Cons

,数据集,RMSE,MAE,R2,MAPE(%),Pearson相关系数,Pearson_p值,Spearman相关系数,Spearman_p值,样本量
0,训练集,2.9754,2.3679,-0.0000,126.8424,NaN,NaN,NaN,NaN,195
1,验证集,2.9162,2.0779,-0.0000,101.3268,NaN,NaN,NaN,NaN,49
2,OOT,2.9962,2.3949,-0.0193,133.9121,NaN,NaN,NaN,NaN,2756


Excel: ./output/xgb_reg\04_model_report\XGBoost_回归模型评估报告.xlsx
HTML:  ./output/xgb_reg\04_model_report\XGBoost_回归模型评估报告.html


In [9]:
print('=== 真实值分桶（三集合，训练集切点）===')
display(reg_result['sheets']['真实值分桶'])

print('=== 预测值分桶（三集合，训练集切点）===')
display(reg_result['sheets']['预测值分桶'])

print('=== 分桶矩阵（三集合合并，含 MAPE + 样本数/占比）===')
display(reg_result['sheets']['分桶矩阵'])

=== 真实值分桶（三集合，训练集切点）===


,数据集,分桶,分箱区间,样本数,样本占比,真实最小值,真实最大值,真实均值,预测均值,预测最低值,预测最高值,误差(预测-真实),MAE,MAPE(%),1值占比
0,训练集,1,"[-inf, -3.9238]",20,0.1026,-7.8595,-3.9243,-5.0476,-0.1884,-0.1884,-0.1884,4.8592,4.8592,96.1417,0.0000
1,训练集,2,"(-3.9238, -2.4685]",19,0.0974,-3.9229,-2.4812,-3.0431,-0.1884,-0.1884,-0.1884,2.8547,2.8548,93.7064,0.0000
2,训练集,3,"(-2.4685, -1.8621]",20,0.1026,-2.4653,-1.8660,-2.1607,-0.1884,-0.1884,-0.1884,1.9723,1.9723,91.2068,0.0000
3,训练集,4,"(-1.8621, -1.1502]",19,0.0974,-1.8466,-1.1591,-1.5393,-0.1884,-0.1884,-0.1884,1.3509,1.3509,87.5301,0.0000
4,训练集,5,"(-1.1502, -0.5056]",20,0.1026,-1.1442,-0.5056,-0.7808,-0.1884,-0.1884,-0.1884,0.5924,0.5924,73.9350,0.0000
5,训练集,6,"(-0.5056, 0.364]",19,0.0974,-0.4564,0.3555,-0.0055,-0.1884,-0.1884,-0.1884,-0.1829,0.2440,382.1725,0.5263
6,训练集,7,"(0.364, 1.2259]",19,0.0974,0.3769,1.1651,0.6889,-0.1884,-0.1884,-0.1884,-0.8773,0.8773,130.3315,1.0000
7,训练集,8,"(1.2259, 2.4956]",20,0.1026,1.2411,2.4818,1.6392,-0.1884,-0.1884,-0.1884,-1.8276,1.8276,111.9930,1.0000
8,训练集,9,"(2.4956, 4.0844]",19,0.0974,2.5511,4.0843,3.1869,-0.1884,-0.1884,-0.1884,-3.3753,3.3753,106.0239,1.0000
9,训练集,10,"(4.0844, +inf]",20,0.1026,4.0845,7.5754,5.3800,-0.1884,-0.1884,-0.1884,-5.5684,5.5684,103.6608,1.0000


=== 预测值分桶（三集合，训练集切点）===


,数据集,分桶,分箱区间,样本数,样本占比,真实最小值,真实最大值,真实均值,预测均值,预测最低值,预测最高值,误差(预测-真实),MAE,MAPE(%),1值占比


=== 分桶矩阵（三集合合并，含 MAPE + 样本数/占比）===


,真实值区间\预测值区间,合计
0,── 训练集 — MAPE矩阵（行=真实值桶，列=预测值桶） ──,
1,"[-inf, -3.9238]",96.14
2,"(-3.9238, -2.4685]",93.71
3,"(-2.4685, -1.8621]",91.21
4,"(-1.8621, -1.1502]",87.53
...,...,...
150,"(0.364, 1.2259]",1.0
151,"(1.2259, 2.4956]",1.0
152,"(2.4956, 4.0844]",1.0
153,"(4.0844, +inf]",1.0


In [10]:
# 自定义切点分桶
custom_bins      = [-3.0, -1.5, -0.5, 0.5, 1.5, 3.0]
custom_bins_pred = [-3.0, -1.5, -0.5, 0.5, 1.5, 3.0]

reg_result_custom = reporter.regression_report(
    datasets_reg,
    filename='XGBoost_回归评估_自定义分桶',
    bins=custom_bins,
    bins_pred=custom_bins_pred,
)
print('=== 自定义切点 — 真实值分桶 ===')
display(reg_result_custom['sheets']['真实值分桶'])
print('=== 自定义切点 — 预测值分桶 ===')
display(reg_result_custom['sheets']['预测值分桶'])
print('=== 自定义切点 — 分桶矩阵 ===')
display(reg_result_custom['sheets']['分桶矩阵'])

C:\Data\工作\AI_model\ml_tool\report.py:1374: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_r,  pearson_p  = pearsonr(y_true, y_pred)
C:\Data\工作\AI_model\ml_tool\report.py:1375: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_r, spearman_p = spearmanr(y_true, y_pred)
C:\Data\工作\AI_model\ml_tool\report.py:1374: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_r,  pearson_p  = pearsonr(y_true, y_pred)
C:\Data\工作\AI_model\ml_tool\report.py:1375: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_r, spearman_p = spearmanr(y_true, y_pred)
C:\Data\工作\AI_model\ml_tool\report.py:1374: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_r,  pearson_p  = pearsonr(y_true, y_pred)
C:\Data\工作\AI_model\ml_tool\report.py:1375: Cons

=== 自定义切点 — 真实值分桶 ===


,数据集,分桶,分箱区间,样本数,样本占比,真实最小值,真实最大值,真实均值,预测均值,预测最低值,预测最高值,误差(预测-真实),MAE,MAPE(%)
0,训练集,1,"[-inf, -3.0]",31,0.1590,-7.8595,-3.0422,-4.4309,-0.1884,-0.1884,-0.1884,4.2425,4.2426,95.4759
1,训练集,2,"(-3.0, -1.5]",39,0.2000,-2.9057,-1.5226,-2.1316,-0.1884,-0.1884,-0.1884,1.9432,1.9432,90.8668
2,训练集,3,"(-1.5, -0.5]",28,0.1436,-1.4833,-0.5056,-0.9413,-0.1884,-0.1884,-0.1884,0.7529,0.7530,77.3430
3,训练集,4,"(-0.5, 0.5]",22,0.1128,-0.4564,0.4146,0.0487,-0.1884,-0.1884,-0.1884,-0.2371,0.2898,350.2609
4,训练集,5,"(0.5, 1.5]",25,0.1282,0.5023,1.4987,0.9531,-0.1884,-0.1884,-0.1884,-1.1415,1.1415,122.4109
5,训练集,6,"(1.5, 3.0]",20,0.1026,1.5883,2.9974,2.2973,-0.1884,-0.1884,-0.1884,-2.4857,2.4856,108.6223
6,训练集,7,"(3.0, +inf]",30,0.1538,3.1867,7.5754,4.7692,-0.1884,-0.1884,-0.1884,-4.9576,4.9576,104.2215
7,训练集,All,All,195,1.0000,-7.8595,7.5754,-0.1689,-0.1884,-0.1884,-0.1884,-0.0195,2.3679,126.8424
8,验证集,1,"[-inf, -3.0]",7,0.1429,-8.1939,-3.3508,-4.7481,-0.1884,-0.1884,-0.1884,4.5597,4.5598,95.6469
9,验证集,2,"(-3.0, -1.5]",7,0.1429,-2.6755,-1.5086,-1.9844,-0.1884,-0.1884,-0.1884,1.7960,1.7960,90.2072


=== 自定义切点 — 预测值分桶 ===


,数据集,分桶,分箱区间,样本数,样本占比,真实最小值,真实最大值,真实均值,预测均值,预测最低值,预测最高值,误差(预测-真实),MAE,MAPE(%)
0,训练集,4,"(-0.5, 0.5]",195,1.0,-7.8595,7.5754,-0.1689,-0.1884,-0.1884,-0.1884,-0.0195,2.3679,126.8424
1,训练集,All,All,195,1.0,-7.8595,7.5754,-0.1689,-0.1884,-0.1884,-0.1884,-0.0195,2.3679,126.8424
2,验证集,4,"(-0.5, 0.5]",49,1.0,-8.1939,8.5034,-0.1942,-0.1884,-0.1884,-0.1884,0.0058,2.0779,101.3268
3,验证集,All,All,49,1.0,-8.1939,8.5034,-0.1942,-0.1884,-0.1884,-0.1884,0.0058,2.0779,101.3268
4,OOT,4,"(-0.5, 0.5]",2756,1.0,-11.5448,11.4595,-0.6011,-0.1884,-0.1884,-0.1884,0.4127,2.3949,133.9121
5,OOT,All,All,2756,1.0,-11.5448,11.4595,-0.6011,-0.1884,-0.1884,-0.1884,0.4127,2.3949,133.9121


=== 自定义切点 — 分桶矩阵 ===


,真实值区间\预测值区间,"(-0.5, 0.5]",合计
0,── 训练集 — MAPE矩阵（行=真实值桶，列=预测值桶） ──,,
1,"[-inf, -3.0]",95.48,95.48
2,"(-3.0, -1.5]",90.87,90.87
3,"(-1.5, -0.5]",77.34,77.34
4,"(-0.5, 0.5]",350.26,350.26
...,...,...,...
84,"(-0.5, 0.5]",0.1263,0.1263
85,"(0.5, 1.5]",0.1125,0.1125
86,"(1.5, 3.0]",0.1274,0.1274
87,"(3.0, +inf]",0.1136,0.1136


## 6. 特征分析报告

In [11]:
analysis_path = reporter.feature_analysis_report(
    df=df,
    features=feature_cols,
    target_col='y_reg',
    dataset_col='dataset',
    month_col='month',
    filename='特征分析报告',
)
print('特征分析报告:', analysis_path)
import openpyxl
wb = openpyxl.load_workbook(analysis_path)
print('sheets:', wb.sheetnames)

特征分析报告: ./output/xgb_reg\01_feature_analysis\特征分析报告.xlsx


sheets: ['整体', 'by_dataset', 'by_month', '特征分箱图']


## 7. 模型保存与加载

训练时传 `save_dir` 自动保存，目录下主要文件：
- `model.pkl` — joblib 格式，含模型对象、入模特征、最优参数
- `tuning_log.xlsx` — 调参日志

支持两种加载方式：
1. `ModelTrainer.load()` — 通过 ml_tool 加载，保持完整接口
2. `joblib.load()` — 直接加载原生模型对象，适合生产打分

In [12]:
import os, joblib

# 查看保存的子目录结构
print('保存目录结构:')
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in sorted(files):
        print(f'{indent}  {f}')

# 方式一：ModelTrainer.load（保持完整接口）
loaded = ModelTrainer.load(OUTPUT_DIR)
pred_via_trainer = loaded.predict(X_oot)
print('\n[方式一] 入模特征:', loaded.selected_features)
print('[方式一] 预测前5个值:', pred_via_trainer[:5].round(4))

# 方式二：joblib 直接加载原生模型，适合生产环境
payload  = joblib.load(os.path.join(OUTPUT_DIR, '05_model_deploy', 'model.pkl'))
model    = payload['model']
features = payload['selected_features']
best_p   = payload['best_params']

print('\n[方式二] model_type:', payload['model_type'])
print('[方式二] 入模特征:', features)
print('[方式二] 最优参数（部分）:', {k: best_p[k] for k in list(best_p)[:3]})

pred_direct = model.predict(X_oot[features])
print('[方式二] 预测前5个值:', pred_direct[:5].round(4))

# 两种方式结果一致验证（浮点精度容忍）
max_diff = float(np.abs(pred_via_trainer - pred_direct).max())
print(f'两种方式最大偏差: {max_diff:.8f}')
assert max_diff < 1e-5, f'预测不一致，最大偏差 {max_diff}'
print('验证通过')


保存目录结构:
xgb_reg/
  01_feature_analysis/
    特征分析报告.xlsx
  02_feature_selection/
    特征筛选报告.xlsx
  03_model_tuning/
    tuning_log.xlsx
  04_model_report/
    XGBoost_回归模型评估报告.html
    XGBoost_回归模型评估报告.xlsx
    XGBoost_回归评估_自定义分桶.html
    XGBoost_回归评估_自定义分桶.xlsx
  05_model_deploy/
    model.pkl

[方式一] 入模特征: ['f4', 'f2', 'f1', 'f3', 'f5']
[方式一] 预测前5个值: [-0.1884 -0.1884 -0.1884 -0.1884 -0.1884]

[方式二] model_type: xgboost_reg
[方式二] 入模特征: ['f4', 'f2', 'f1', 'f3', 'f5']
[方式二] 最优参数（部分）: {'colsample_bytree': 0.8, 'gamma': 7.0, 'learning_rate': 0.23}
[方式二] 预测前5个值: [-0.1884 -0.1884 -0.1884 -0.1884 -0.1884]
两种方式最大偏差: 0.00000000
验证通过
